# Does depth let standard attention catch up to a single Strassen layer?

**Hypothesis** (from the paper's own argument): a *single layer* of Strassen attention can represent the Match3 task's 3rd-order relational predicate directly, while standard (2nd-order, pairwise) attention needs *multiple layers* to approximate the same relation compositionally — and the number of layers it needs should grow as the sequence gets longer (more tokens to relate).

**Experiment**: for several *fixed* sequence lengths (`10, 20, 30, 40`, all with the paper's `M=37`), train:
- 1 Strassen-attention model, **1 layer** (the reference/target performance)
- 4 standard-attention models, **depths 1, 2, 3, 4**

...then compare each standard-attention depth's validation accuracy against the single-layer Strassen reference, for every length. If the hypothesis holds, the depth at which standard attention "catches up" to Strassen should increase with sequence length.

Unlike `match3_strassen_vs_standard_own_stack.ipynb`, this notebook uses **fixed-length** sequences per dataset (`min_len == seq_len`), so length is a controlled variable rather than a range — and it's a **fast sweep**, not paper-scale: with `4 lengths x 5 models = 20` training runs, each uses a smaller dataset and fewer epochs so the whole sweep finishes in reasonable time. Scale up `NUM_TRAIN`/`NUM_EPOCHS` in the config cell if the trend looks interesting and you want more confidence in it.

This notebook does not clone or depend on the original `strassen-attention-neurips25` code — it uses this repo's own `datasets/` and `models/` (same as `match3_strassen_vs_standard_own_stack.ipynb`).

## 1. Setup

In [1]:
import os

REPO_URL = "https://github.com/MMVahedi/Thesis.git"  # use a token-embedded URL here if the repo is private
REPO_DIR = "Thesis"


def already_in_repo() -> bool:
    # True once we're inside the repo (whether from a previous run's %cd in
    # this same session, or because the runtime happened to start here) —
    # checking this, rather than just os.path.isdir(REPO_DIR), is what makes
    # this cell safe to re-run without re-cloning into a nested Thesis/Thesis.
    return os.path.isdir("datasets") and os.path.isdir("models")


if not already_in_repo():
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL}
    %cd {REPO_DIR}


Cloning into 'Thesis'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 79 (delta 21), reused 74 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 8.64 MiB | 34.17 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/Thesis


In [ ]:
!pip install -q opt_einsum scikit-learn


In [ ]:
import sys, os, json, random, time, gc
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

sys.path.insert(0, ".")  # ensure repo-root imports (compgen) resolve when the kernel starts elsewhere

# This notebook defaults to DTYPE=torch.float64 (the paper's setting, see the
# config cell below) — Apple's MPS backend does not support float64 at all,
# so we don't select it here even if available.
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    if torch.backends.mps.is_available():
        print("NOTE: MPS (Apple GPU) is available, but this notebook defaults to "
              "DTYPE=torch.float64, which MPS cannot run. Defaulting to CPU.")

print("Device:", device)
if device == "cpu":
    print("Running on CPU. These models are small (hidden_dim=128), so it's usable, "
          "just slower than a GPU.")


## 2. Config

`LENGTHS` are the fixed sequence lengths swept over (every generated sequence at a given length is exactly that long: `min_len == seq_len`). `STANDARD_DEPTHS` is `num_layers` for the four standard-attention models; `STRASSEN_DEPTH` is fixed at 1. Model/data hyperparameters (`M`, `HIDDEN_DIM`, `DROPOUT_RATE`, etc.) match the paper's own `hyperparams.json` "33" experiment — only dataset size and epoch count are reduced for a fast sweep.

Optimization hyperparameters (`LR`, `WEIGHT_DECAY`, `NUM_EPOCHS`) are kept in a separate `OPTIM_PARAMS` dict, one entry per model (`"strassen"`, `"standard_depth1"`, ...), so each model can be tuned independently instead of sharing one global setting.

In [ ]:
SEEDS = [42, 23, 54, 96, 12, 48, 67, 33]
MASTER_SEED = 45   # seeds dataset generation, so every length's data is reproducible
SEED_INDEX = 0     # which entry of SEEDS to train each model with

LENGTHS = [10, 15, 20, 25]
STANDARD_DEPTHS = [1, 2, 4, 6]
STRASSEN_DEPTH = 1

config = {
    # --- task / data ---
    "M": 31,
    "NUM_BINS": 4,
    "NUM_TRAIN": 10000,    # fast sweep; paper uses 45000
    "NUM_VAL": 1000,      # fast sweep; paper uses 5000
    "SHUFFLE": True,
    "NUM_WORKERS": 2,

    # --- model (hyperparams.json "33") ---
    "HIDDEN_DIM": 128,
    "NUM_HEADS": 4,
    "DROPOUT_RATE": 0.4,
    "SHARE_LAYERS": False,
    "USE_NORM": False,
    "USE_ATTENTION_DROPOUT": False,

    "BATCH_SIZE": 2048,
    "DTYPE": torch.float32,

    "DEVICE": device,
    "SEEDS": SEEDS,
    "SEED_INDEX": SEED_INDEX,
    "MASTER_SEED": MASTER_SEED,
}
config["EMBEDDING_NORM_SCALAR"] = config["M"]

# --- optimization, kept separate per model so each can be tuned independently ---
# (all default to the paper's hyperparams.json "33" values; edit any entry below
# to give that specific model a different LR / weight decay / epoch budget)
MODEL_NAMES = ["strassen"] + [f"standard_depth{d}" for d in STANDARD_DEPTHS]
OPTIM_PARAMS = {
    name: {
        "LR": 1e-3,
        "WEIGHT_DECAY": 0,
        "NUM_EPOCHS": 100,     # fast sweep; paper uses 500
    }
    for name in MODEL_NAMES
}

SAVE_EVERY = 10   # epochs between checkpoint saves, so an interrupted run doesn't lose everything
RESULTS_DIR = "results/match3_depth_vs_strassen"

{k: (str(v) if k == "DTYPE" else v) for k, v in config.items()}


## 3. Data

Builds one train/val pair *per length* — every sequence in a given length's dataset is exactly that long (`min_len = seq_len = length`).

In [ ]:
from compgen.datasets.generators.match3 import Match3Config, Match3Generator
from compgen.datasets.torch_datasets.match3 import Match3Dataset, match3_collate_fn

random.seed(config["MASTER_SEED"])
np.random.seed(config["MASTER_SEED"])
torch.manual_seed(config["MASTER_SEED"])

def build_fixed_length_split(length: int, num_instances: int) -> Match3Dataset:
    gen_config = Match3Config(
        num_instances=num_instances,
        seq_len=length,
        min_len=length,   # fixed length: every sequence is exactly `length` long
        M=config["M"],
        num_bins=config["NUM_BINS"],
    )
    path = f"/tmp/match3_len{length}_{num_instances}.jsonl"
    Match3Generator(gen_config).save(path)
    return Match3Dataset(path)

datasets_by_length = {}
for length in LENGTHS:
    datasets_by_length[length] = {
        "train": build_fixed_length_split(length, config["NUM_TRAIN"]),
        "val": build_fixed_length_split(length, config["NUM_VAL"]),
    }

{length: {name: len(ds) for name, ds in splits.items()} for length, splits in datasets_by_length.items()}


In [ ]:
dataloaders_by_length = {
    length: {
        name: DataLoader(
            ds,
            batch_size=config["BATCH_SIZE"],
            shuffle=config["SHUFFLE"],
            collate_fn=match3_collate_fn,
            num_workers=config["NUM_WORKERS"],
        )
        for name, ds in splits.items()
    }
    for length, splits in datasets_by_length.items()
}


## 4. Training loop

Same `run_epoch` as the other notebook (unifies train/eval into one function, avoiding the original repo's dropout-never-re-enabled bug). `run_model` takes `num_layers` and an `optim_params` dict (`LR`, `WEIGHT_DECAY`, `NUM_EPOCHS`) explicitly, since both are swept/varied per model. After training, `run_model` deletes the model and optimizer and runs garbage collection (plus `torch.cuda.empty_cache()`) so GPU VRAM is freed before the next model is trained — with ~20 models trained in sequence on limited VRAM, this keeps peak memory bounded to one model at a time.

There's no held-out test set in this notebook, only train/val, so `plot_history` (defined below) plots each finished model's train/val loss and accuracy curves and reports its final validation accuracy — that's called right after each `run_model` call in section 5.

In [ ]:
import matplotlib.pyplot as plt
from compgen.models.tasks.match3 import Match3Model

def run_epoch(model, dataloader, criterion, device, dtype, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, accuracies, f1_scores = 0.0, [], []
    with torch.enable_grad() if is_train else torch.no_grad():
        for batch in dataloader:
            if is_train:
                optimizer.zero_grad()

            y_hat, _ = model(batch)
            target = batch["label"].view(-1, 1).to(dtype).to(device)
            y_hat = y_hat.reshape(len(target), -1).to(dtype).to(device)
            mask = target != -100

            loss = criterion(y_hat[mask], target[mask])
            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            preds = y_hat > 0.5
            accuracies.append((preds[mask] == target[mask]).float().mean().item())
            f1_scores.append(f1_score(
                target[mask].detach().cpu().numpy(),
                preds[mask].detach().cpu().numpy(),
                zero_division=0,
            ))

    return total_loss / len(dataloader), np.mean(accuracies), np.mean(f1_scores)


def free_gpu_memory():
    """Reclaim VRAM held by the caching allocator once no more live tensors reference it."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def plot_history(history, title):
    """Plot train/val loss and accuracy curves for one finished training run, and report its final val accuracy."""
    epochs = range(1, len(history["train"]["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["train"]["loss"], label="train")
    axes[0].plot(epochs, history["val"]["loss"], label="val")
    axes[0].set_title(f"{title} — loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train"]["acc"], label="train")
    axes[1].plot(epochs, history["val"]["acc"], label="val")
    axes[1].set_title(f"{title} — accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"{title}: final val_acc={history['val']['acc'][-1]:.3f}")


def run_model(attention_type, num_layers, length, config, dataloaders, optim_params, results_dir=RESULTS_DIR):
    seed = config["SEEDS"][config["SEED_INDEX"]]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = Match3Model(
        hidden_dim=config["HIDDEN_DIM"],
        attention_type=attention_type,
        num_layers=num_layers,
        num_heads=config["NUM_HEADS"],
        dropout_rate=config["DROPOUT_RATE"],
        embedding_norm_scalar=config["EMBEDDING_NORM_SCALAR"],
        use_norm=config["USE_NORM"],
        use_attention_dropout=config["USE_ATTENTION_DROPOUT"],
        share_layers=config["SHARE_LAYERS"],
        dtype=config["DTYPE"],
        device=config["DEVICE"],
    ).to(config["DEVICE"])

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=optim_params["LR"], weight_decay=optim_params["WEIGHT_DECAY"]
    )
    criterion = torch.nn.BCELoss()

    history = {"train": {"loss": [], "acc": [], "f1": []}, "val": {"loss": [], "acc": [], "f1": []}}
    model_name = f"{attention_type}_depth{num_layers}"
    os.makedirs(results_dir, exist_ok=True)
    history_path = os.path.join(results_dir, f"length{length}_{model_name}_history.json")

    num_epochs = optim_params["NUM_EPOCHS"]
    for epoch in range(num_epochs):
        t0 = time.time()
        loss, acc, f1 = run_epoch(
            model, dataloaders["train"], criterion, config["DEVICE"], config["DTYPE"], optimizer=optimizer
        )
        history["train"]["loss"].append(loss)
        history["train"]["acc"].append(acc)
        history["train"]["f1"].append(f1)

        loss, acc, f1 = run_epoch(model, dataloaders["val"], criterion, config["DEVICE"], config["DTYPE"])
        history["val"]["loss"].append(loss)
        history["val"]["acc"].append(acc)
        history["val"]["f1"].append(f1)

        if epoch % SAVE_EVERY == 0 or epoch == num_epochs - 1:
            with open(history_path, "w") as f:
                json.dump(history, f)

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(
                f"[length={length} {model_name}] epoch {epoch + 1}/{num_epochs} "
                f"({time.time() - t0:.1f}s) "
                f"train_acc={history['train']['acc'][-1]:.3f} "
                f"val_acc={history['val']['acc'][-1]:.3f}"
            )

    # Training for this model is done — drop our own references to the model/optimizer
    # (the only ones keeping their GPU tensors alive) before reclaiming VRAM, so the
    # next model trained in this notebook starts with a clean slate.
    del model, optimizer
    free_gpu_memory()

    return history


### Quick timing check (optional but recommended)

Runs 1 epoch at the largest length (most expensive, both for standard and Strassen) to estimate total sweep runtime before committing to the full `NUM_EPOCHS x 20 models`.

In [ ]:
_longest = max(LENGTHS)
_probe_optim_params = dict(OPTIM_PARAMS["strassen"], NUM_EPOCHS=1)
_t0 = time.time()
run_model("strassen", STRASSEN_DEPTH, _longest, config, dataloaders_by_length[_longest],
          optim_params=_probe_optim_params, results_dir=f"{RESULTS_DIR}/_probe")
_epoch_time = time.time() - _t0
_total_models = len(LENGTHS) * (1 + len(STANDARD_DEPTHS))
_avg_epochs = np.mean([p["NUM_EPOCHS"] for p in OPTIM_PARAMS.values()])
print(f"~{_epoch_time:.1f}s/epoch at length={_longest} (Strassen, most expensive case) -> "
      f"~{_epoch_time * _avg_epochs * _total_models / 60:.1f} min upper bound for "
      f"the full sweep ({_total_models} models, ~{_avg_epochs:.0f} epochs avg per OPTIM_PARAMS; "
      f"shorter lengths and standard attention are both cheaper than this estimate, and "
      f"per-model NUM_EPOCHS may differ)")


## 5. Run the full sweep

For each length: train 1 Strassen model (`STRASSEN_DEPTH` layers) and 4 standard models (`STANDARD_DEPTHS` layers each). Each model trains in its own cell below, using its own entry from `OPTIM_PARAMS`, so any model can be re-run, retuned, or skipped independently of the others. `run_model` frees the model/optimizer from GPU VRAM at the end of each cell before the next one runs.

In [ ]:
results = {length: {} for length in LENGTHS}


### length = 10 (`LENGTHS[0]`)

In [ ]:
length = LENGTHS[0]
print(f"\n=== length={length}: strassen (depth={STRASSEN_DEPTH}) ===")
results[length]["strassen"] = run_model(
    "strassen", STRASSEN_DEPTH, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS["strassen"],
)
plot_history(results[length]["strassen"], f"length={length} strassen")


In [ ]:
length = LENGTHS[0]
depth = STANDARD_DEPTHS[0]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[0]
depth = STANDARD_DEPTHS[1]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[0]
depth = STANDARD_DEPTHS[2]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[0]
depth = STANDARD_DEPTHS[3]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


### length = 15 (`LENGTHS[1]`)

In [ ]:
length = LENGTHS[1]
print(f"\n=== length={length}: strassen (depth={STRASSEN_DEPTH}) ===")
results[length]["strassen"] = run_model(
    "strassen", STRASSEN_DEPTH, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS["strassen"],
)
plot_history(results[length]["strassen"], f"length={length} strassen")


In [ ]:
length = LENGTHS[1]
depth = STANDARD_DEPTHS[0]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[1]
depth = STANDARD_DEPTHS[1]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[1]
depth = STANDARD_DEPTHS[2]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[1]
depth = STANDARD_DEPTHS[3]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


### length = 20 (`LENGTHS[2]`)

In [ ]:
length = LENGTHS[2]
print(f"\n=== length={length}: strassen (depth={STRASSEN_DEPTH}) ===")
results[length]["strassen"] = run_model(
    "strassen", STRASSEN_DEPTH, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS["strassen"],
)
plot_history(results[length]["strassen"], f"length={length} strassen")


In [ ]:
length = LENGTHS[2]
depth = STANDARD_DEPTHS[0]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[2]
depth = STANDARD_DEPTHS[1]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[2]
depth = STANDARD_DEPTHS[2]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[2]
depth = STANDARD_DEPTHS[3]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


### length = 25 (`LENGTHS[3]`)

In [ ]:
length = LENGTHS[3]
print(f"\n=== length={length}: strassen (depth={STRASSEN_DEPTH}) ===")
results[length]["strassen"] = run_model(
    "strassen", STRASSEN_DEPTH, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS["strassen"],
)
plot_history(results[length]["strassen"], f"length={length} strassen")


In [ ]:
length = LENGTHS[3]
depth = STANDARD_DEPTHS[0]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[3]
depth = STANDARD_DEPTHS[1]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[3]
depth = STANDARD_DEPTHS[2]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


In [ ]:
length = LENGTHS[3]
depth = STANDARD_DEPTHS[3]
model_name = f"standard_depth{depth}"
print(f"\n=== length={length}: {model_name} ===")
results[length][model_name] = run_model(
    "standard", depth, length, config, dataloaders_by_length[length],
    optim_params=OPTIM_PARAMS[model_name],
)
plot_history(results[length][model_name], f"length={length} {model_name}")


## 6. Compare: does depth catch up to Strassen?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(LENGTHS), figsize=(5 * len(LENGTHS), 4), sharey=True)
if len(LENGTHS) == 1:
    axes = [axes]

for ax, length in zip(axes, LENGTHS):
    strassen_acc = results[length]["strassen"]["val"]["acc"][-1]
    standard_accs = [results[length][f"standard_depth{d}"]["val"]["acc"][-1] for d in STANDARD_DEPTHS]

    ax.plot(STANDARD_DEPTHS, standard_accs, "o-", color="tab:orange", label="Standard attention")
    ax.axhline(strassen_acc, color="tab:blue", linestyle="--",
               label=f"Strassen ({STRASSEN_DEPTH} layer)")
    ax.set_title(f"length={length}")
    ax.set_xlabel("standard attention depth (layers)")
    ax.set_xticks(STANDARD_DEPTHS)
    ax.set_ylim(0, 1.05)

axes[0].set_ylabel("final validation accuracy")
axes[0].legend()
plt.tight_layout()
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, "depth_vs_strassen_by_length.png"), dpi=150)
plt.show()


In [ ]:
import pandas as pd

rows = []
for length in LENGTHS:
    row = {"length": length, "strassen_acc": results[length]["strassen"]["val"]["acc"][-1]}
    for d in STANDARD_DEPTHS:
        row[f"standard_depth{d}_acc"] = results[length][f"standard_depth{d}"]["val"]["acc"][-1]
    rows.append(row)
pd.DataFrame(rows)


## 7. Depth needed to catch up, vs. length

For each length, the smallest standard-attention depth whose validation accuracy reaches (or exceeds) the Strassen reference — `None` if no tested depth caught up. If the hypothesis holds, this should trend upward with length.

In [ ]:
depth_needed = {}
for length in LENGTHS:
    strassen_acc = results[length]["strassen"]["val"]["acc"][-1]
    needed = None
    for d in STANDARD_DEPTHS:
        if results[length][f"standard_depth{d}"]["val"]["acc"][-1] >= strassen_acc:
            needed = d
            break
    depth_needed[length] = needed

print("Depth needed to catch up to single-layer Strassen, per length:")
for length, d in depth_needed.items():
    print(f"  length={length}: {d if d is not None else f'none of {STANDARD_DEPTHS} caught up'}")

plottable = {l: d for l, d in depth_needed.items() if d is not None}
if plottable:
    plt.figure(figsize=(6, 4))
    plt.plot(list(plottable.keys()), list(plottable.values()), "o-", color="tab:green")
    plt.xlabel("sequence length")
    plt.ylabel("min. standard-attention depth to catch up")
    plt.ylim(0, max(STANDARD_DEPTHS) + 1)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "depth_needed_vs_length.png"), dpi=150)
    plt.show()
else:
    print("No length had a standard-attention depth catch up within", STANDARD_DEPTHS,
          "- consider raising NUM_EPOCHS or STANDARD_DEPTHS.")


## 8. (Optional) Download results

In [ ]:
!zip -rq match3_depth_vs_strassen_results.zip {RESULTS_DIR}

try:
    from google.colab import files
    files.download("match3_depth_vs_strassen_results.zip")
except ImportError:
    print("Not on Colab — find match3_depth_vs_strassen_results.zip in the working directory.")
